In [7]:
import numpy as np
import sys , os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.join(os.getcwd(),'..','src'))

from decision_trees import DecisionTree
from utils import accuracy, precision, recall, f1_score, confusion_matrix, print_tree

X_train_bc = np.load('../Data/processed/bc_X_train.npy')
X_test_bc = np.load('../Data/processed/bc_X_test.npy')
y_train_bc = np.load('../Data/processed/bc_y_train.npy')
y_test_bc = np.load('../Data/processed/bc_y_test.npy') 

X_train_hd = np.load('../Data/processed/hd_X_train.npy')
y_train_hd = np.load('../Data/processed/hd_y_train.npy')
X_test_hd = np.load('../Data/processed/hd_X_test.npy')
y_test_hd = np.load('../Data/processed/hd_y_test.npy')

print('BC train:', X_train_bc.shape, 'HD train:', X_train_hd.shape)

BC train: (455, 30) HD train: (820, 13)


In [8]:
bc_idx_1 = np.where(y_train_bc == 1)[0]
bc_idx_0 = np.where(y_train_bc == 0)[0]

np.random.seed(42)
keep_0 = np.random.choice(bc_idx_0, size=max(1, len(bc_idx_0) // 10), replace=False)
imbalanced_idx = np.concatenate([bc_idx_1, keep_0])
np.random.shuffle(imbalanced_idx)

X_imb = X_train_bc[imbalanced_idx]
y_imb = y_train_bc[imbalanced_idx]

print('Imbalanced class counts:')
print('  class 0 (benign)  :', np.sum(y_imb == 0))
print('  class 1 (malignant):', np.sum(y_imb == 1))

m_imb = DecisionTree(criterion='gini', max_depth=5, task='classification')
m_imb.fit(X_imb, y_imb)
pred_imb = m_imb.predict(X_test_bc)

print('\nMetrics on imbalanced model:')
print('accuracy :', accuracy(y_test_bc, pred_imb))
print('precision:', precision(y_test_bc, pred_imb))
print('recall   :', recall(y_test_bc, pred_imb))
print('f1       :', f1_score(y_test_bc, pred_imb))
print('\nConfusion matrix:')
print(confusion_matrix(y_test_bc, pred_imb))
print('\nNote: accuracy looks acceptable but recall on class 0 is poor — the model is biased toward the majority class.')

Imbalanced class counts:
  class 0 (benign)  : 28
  class 1 (malignant): 169

Metrics on imbalanced model:
accuracy : 0.7982456140350878
precision: 0.6515151515151515
recall   : 1.0
f1       : 0.7889908256880734

Confusion matrix:
[[48 23]
 [ 0 43]]

Note: accuracy looks acceptable but recall on class 0 is poor — the model is biased toward the majority class.


In [11]:
np.random.seed(42)
noise_train = np.random.randn(X_train_bc.shape[0], 50)
noise_test = np.random.randn(X_test_bc.shape[0], 50)

X_train_noisy = np.hstack([X_train_bc, noise_train])
X_test_noisy = np.hstack([X_test_bc, noise_test])

m_orig = DecisionTree(criterion='gini', max_depth=5, task='classification')
m_orig.fit(X_train_bc, y_train_bc)
acc_orig = accuracy(y_test_bc, m_orig.predict(X_test_bc))

m_noisy = DecisionTree(criterion='gini', max_depth=5, task='classification')
m_noisy.fit(X_train_noisy, y_train_bc)
acc_noisy = accuracy(y_test_bc, m_noisy.predict(X_test_noisy))

print('Original (30 features) test accuracy :', acc_orig)
print('Noisy    (80 features) test accuracy  :', acc_noisy)
print('\nIf accuracy drops, the greedy scan wasted early splits on noise features.')
print('Trees have no built-in feature selection — a noisy feature can win a split if it happens to score well on that node.')

Original (30 features) test accuracy : 0.9385964912280702
Noisy    (80 features) test accuracy  : 0.9385964912280702

If accuracy drops, the greedy scan wasted early splits on noise features.
Trees have no built-in feature selection — a noisy feature can win a split if it happens to score well on that node.


In [17]:

i = 0   
j = 2  

keep_cols = [c for c in range(X_train_bc.shape[1]) if c != j]

m_both = DecisionTree(criterion='gini', max_depth=5, task='classification')
m_both.fit(X_train_bc, y_train_bc)
acc_both = accuracy(y_test_bc, m_both.predict(X_test_bc))

m_one = DecisionTree(criterion='gini', max_depth=5, task='classification')
m_one.fit(X_train_bc[:, keep_cols], y_train_bc)
acc_one = accuracy(y_test_bc, m_one.predict(X_test_bc[:, keep_cols]))

print('\nTest accuracy (both correlated features)      :', acc_both)
print('Test accuracy (one correlated feature removed):', acc_one)

print('\nTree root split with both features:')
print_tree(m_both.root, depth=0, feature_names=[f'feature[{k}]' for k in range(X_train_bc.shape[1])])

print('\nTree root split with one feature removed:')
print_tree(m_one.root, depth=0, feature_names=[f'feature[{k}]' for k in keep_cols])


Test accuracy (both correlated features)      : 0.9385964912280702
Test accuracy (one correlated feature removed): 0.9385964912280702

Tree root split with both features:
If feature[7] < 0.05128:
  If feature[20] < 16.83:
    If feature[10] < 0.62555:
      If feature[24] < 0.17765:
        If feature[14] < 0.003309:
          Leaf: value = 0
        Else:
          Leaf: value = 0
      Else:
        Leaf: value = 1
    Else:
      If feature[4] < 0.09068:
        Leaf: value = 0
      Else:
        Leaf: value = 1
  Else:
    If feature[1] < 16.189999999999998:
      Leaf: value = 0
    Else:
      If feature[17] < 0.010125499999999999:
        Leaf: value = 1
      Else:
        Leaf: value = 0
Else:
  If feature[27] < 0.14655:
    If feature[22] < 115.25:
      If feature[1] < 21.055:
        Leaf: value = 0
      Else:
        Leaf: value = 1
    Else:
      Leaf: value = 1
  Else:
    If feature[16] < 0.13565:
      Leaf: value = 1
    Else:
      Leaf: value = 0

Tree root spli

In [18]:
np.random.seed(42)
tiny_idx = np.random.choice(len(X_train_bc), size=20, replace=False)
X_tiny = X_train_bc[tiny_idx]
y_tiny = y_train_bc[tiny_idx]

m_tiny = DecisionTree(criterion='gini', max_depth=None, task='classification')
m_tiny.fit(X_tiny, y_tiny)

tiny_train_acc = accuracy(y_tiny, m_tiny.predict(X_tiny))
tiny_test_acc = accuracy(y_test_bc, m_tiny.predict(X_test_bc))

print('Small sample (n=20):')
print('  train accuracy:', tiny_train_acc)
print('  test accuracy :', tiny_test_acc)

Small sample (n=20):
  train accuracy: 1.0
  test accuracy : 0.8771929824561403
